In [1]:
import numpy as np
import mediapipe as mp

In [21]:
import json
import os

# Load JSON
with open(r'C:\Users\justi\WLASL\start_kit\WLASL_v0.3.json', 'r') as f:
    data = json.load(f)

# Build lookup dict: video_id -> gloss
video_to_gloss = {}
for entry in data:
    gloss = entry['gloss']
    for instance in entry['instances']:
        video_id = instance['video_id']
        video_to_gloss[video_id] = gloss

# Now loop over your videos FIRST, then look up
video_folder = r'C:\Users\justi\WLASL\start_kit\raw_videos'

for filename in os.listdir(video_folder):
    video_id = os.path.splitext(filename)[0]  # strip .mp4 -> "69241"
    
    gloss = video_to_gloss.get(video_id)
    
    if gloss:
        print(f"{filename} -> {gloss}")
    else:
        print(f"{filename} -> NOT FOUND")

00414.mp4 -> about
00415.mp4 -> about
00416.mp4 -> about
00426.mp4 -> about
00623.mp4 -> accident
00624.mp4 -> accident
00625.mp4 -> accident
00626.mp4 -> accident
00627.mp4 -> accident
00628.mp4 -> accident
00629.mp4 -> accident
00639.mp4 -> accident
01383.mp4 -> africa
01384.mp4 -> africa
01385.mp4 -> africa
01386.mp4 -> africa
01387.mp4 -> africa
01388.mp4 -> africa
01398.mp4 -> africa
01433.mp4 -> afternoon
01434.mp4 -> afternoon
01435.mp4 -> afternoon
01436.mp4 -> afternoon
01442.mp4 -> afternoon
01460.mp4 -> again
01461.mp4 -> again
01462.mp4 -> again
01463.mp4 -> again
01464.mp4 -> again
01471.mp4 -> again
01485.mp4 -> age
01486.mp4 -> age
01487.mp4 -> age
01495.mp4 -> age
01986.mp4 -> all
01987.mp4 -> all
01988.mp4 -> all
02003.mp4 -> all
02103.mp4 -> alone
02104.mp4 -> alone
02105.mp4 -> alone
02106.mp4 -> alone
02112.mp4 -> alone
02999.mp4 -> apple
03000.mp4 -> apple
03001.mp4 -> apple
03002.mp4 -> apple
03003.mp4 -> apple
03008.mp4 -> apple
03055.mp4 -> appointment
03056.mp4

In [25]:
from collections import defaultdict
class_counts = defaultdict(int)

for filename in os.listdir(video_folder):
    video_id = os.path.splitext(filename)[0]
    gloss = video_to_gloss.get(video_id)
    if gloss:
        class_counts[gloss] += 1


print("=== Classes with < 10 videos ===")
for gloss, count in sorted(class_counts.items()):
    if count >= 8:
        print(f"{gloss}: {count}")

print("\n=== Summary ===")
print(f"Total classes found: {len(class_counts)}")
print(f"Classes with >= 10: {sum(1 for c in class_counts.values() if c >= 10)}")
print(f"Classes with  < 10: {sum(1 for c in class_counts.values() if c < 10)}")
print(f"Classes with  > 8: {sum(1 for c in class_counts.values() if c >= 8)}")

=== Classes with < 10 videos ===
accident: 8
basketball: 8
before: 10
bird: 8
brother: 9
candy: 8
change: 9
check: 8
cold: 8
computer: 11
cool: 11
corn: 8
cousin: 10
dark: 8
deaf: 8
drink: 8
family: 8
far: 8
government: 8
hot: 8
later: 9
laugh: 8
leave: 8
like: 8
lose: 8
man: 8
new: 8
orange: 8
pizza: 8
play: 9
school: 8
short: 9
tall: 9
thanksgiving: 9
thin: 11
trade: 10
what: 9
white: 8
who: 10
write: 8
yes: 8

=== Summary ===
Total classes found: 334
Classes with >= 10: 7
Classes with  < 10: 327
Classes with  > 8: 41


In [23]:
import shutil
# Get classes with >= 8 videos
valid_classes = {gloss for gloss, count in class_counts.items() if count >= 8}

print(f"Classes with >= 8 videos: {len(valid_classes)}")

# Copy videos into new organized dataset folder
output_folder = r'C:\Users\justi\WLASL\start_kit\dataset_use'

for filename in os.listdir(video_folder):
    video_id = os.path.splitext(filename)[0]
    gloss = video_to_gloss.get(video_id)
    
    if gloss and gloss in valid_classes:
        class_folder = os.path.join(output_folder, gloss)
        os.makedirs(class_folder, exist_ok=True)
        
        src = os.path.join(video_folder, filename)
        dst = os.path.join(class_folder, filename)
        shutil.copy2(src, dst)

print("Done! Dataset created at:", output_folder)

# Verify
print("\n=== Verification ===")
for class_name in sorted(os.listdir(output_folder)):
    count = len(os.listdir(os.path.join(output_folder, class_name)))
    print(f"{class_name}: {count}")

Classes with >= 8 videos: 41
Done! Dataset created at: C:\Users\justi\WLASL\start_kit\dataset_use

=== Verification ===
accident: 8
basketball: 8
before: 10
bird: 8
brother: 9
candy: 8
change: 9
check: 8
cold: 8
computer: 11
cool: 11
corn: 8
cousin: 10
dark: 8
deaf: 8
drink: 8
family: 8
far: 8
government: 8
hot: 8
later: 9
laugh: 8
leave: 8
like: 8
lose: 8
man: 8
new: 8
orange: 8
pizza: 8
play: 9
school: 8
short: 9
tall: 9
thanksgiving: 9
thin: 11
trade: 10
what: 9
white: 8
who: 10
write: 8
yes: 8


In [19]:
import random
random.seed(80)

dataset_folder = r'C:\Users\justi\WLASL\start_kit\dataset_use'
split_folder = r'C:\Users\justi\WLASL\start_kit\dataset_split'

# Create train/val/test subfolders
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(split_folder, split), exist_ok=True)

for class_name in os.listdir(dataset_folder):
    class_path = os.path.join(dataset_folder, class_name)
    if not os.path.isdir(class_path):
        continue

    videos = os.listdir(class_path)
    random.shuffle(videos)

    n = len(videos)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)
    # test gets the remainder to ensure all videos are used
    
    splits = {
        'train': videos[:n_train],
        'val':   videos[n_train:n_train + n_val],
        'test':  videos[n_train + n_val:]
    }

    for split, files in splits.items():
        split_class_folder = os.path.join(split_folder, split, class_name)
        os.makedirs(split_class_folder, exist_ok=True)
        for f in files:
            src = os.path.join(class_path, f)
            dst = os.path.join(split_class_folder, f)
            shutil.copy2(src, dst)

# Verify
print("=== Split Summary ===")
for split in ['train', 'val', 'test']:
    total = sum(
        len(os.listdir(os.path.join(split_folder, split, c)))
        for c in os.listdir(os.path.join(split_folder, split))
    )
    print(f"{split}: {total} videos")

=== Split Summary ===
train: 227 videos
val: 41 videos
test: 85 videos
